(2) Compute similarity

(2.1) text similarity

用 Sentence-BERT 或 TFIDF 计算 overview 的文本相似度

In [2]:
import pandas as pd
df1 = pd.read_csv('cleaned_movies_metadata.csv')
df1.head()
print(df1['original_language'].value_counts().head(10))

original_language
en    5363
Name: count, dtype: int64


SBERT

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import pandas as pd

# Step 1
print("1. Processing overview column...")
df1['overview'] = df1['overview'].fillna('').astype(str)

# Step 2
print("2. Loading Sentence-BERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Step 3
print("3. Generating embeddings...")
overview_list = df1['overview'].tolist()

embeddings = model.encode(
    overview_list,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"Embedding matrix shape: {embeddings.shape}")

# Step 4
print("4. Calculating similarity matrix...")
similarity_matrix = cosine_similarity(embeddings)

print("Done!")
print(similarity_matrix[:5, :5].round(3))


1. Processing overview column...
2. Loading Sentence-BERT model...
3. Generating embeddings...


Batches:   0%|          | 0/168 [00:00<?, ?it/s]

Embedding matrix shape: (5363, 384)
4. Calculating similarity matrix...
Done!
[[ 1.     0.31   0.173  0.041 -0.006]
 [ 0.31   1.     0.292  0.164  0.234]
 [ 0.173  0.292  1.     0.211  0.227]
 [ 0.041  0.164  0.211  1.     0.331]
 [-0.006  0.234  0.227  0.331  1.   ]]


In [4]:
def recommend_movies(title, df, similarity_matrix, top_n=5):

    idx = df[df['title'] == title].index[0]

    scores = list(enumerate(similarity_matrix[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:top_n+1]

    movie_indices = [i[0] for i in scores]

    return df.iloc[movie_indices][['title']]

In [17]:
input_title = "Casino"
re_list = recommend_movies(input_title, df1, similarity_matrix)
print(re_list)

                title
2254  Dante's Inferno
1004   House of Games
885    Guys and Dolls
4652        Cornered!
4354       The System


In [20]:
print(df1[df1['title'] == input_title]['overview'].values[0])

The life of the gambling paradise – Las Vegas – and its dark mafia underbelly.


In [23]:
print(df1[df1['title'].isin(re_list['title'])]['overview'].values)

['In New York, a gambler is challenged to take a cold female missionary to Havana, but they fall for each other, and the bet has a hidden motive to finance a crap game.'
 'A psychiatrist comes to the aid of a compulsive gambler and is led by a smooth-talking grifter into the shadowy but compelling world of stings, scams, and con men.'
 "A carny builds a gambling empire at the expense of his family's wellbeing."
 'A gambling boss is pressured by the law and press when a crusade is started against him after one of his collectors becomes a killer.'
 'During their nightly poker game a group of lowlifes are terrorized in their own convenience store by a masked killer.']


- Advanced content-based filtering using clustering

Although K-Means clustering can reduce computational complexity by grouping similar items, it requires the number of clusters K to be predefined, which may not accurately reflect the true structure of movie genres and semantic relationships.

Therefore, this project directly applied SBERT embeddings with cosine similarity to preserve semantic flexibility between movies.

For larger-scale recommendation systems, Approximate Nearest Neighbor (ANN) search methods such as FAISS or HNSW are commonly used to efficiently retrieve semantically similar items without computing the full similarity matrix.

This project used cosine similarity because the dataset size remained computationally manageable.